# Observability & Debugging — Part 3: Debugging Tool Failures & Context Pressure

This notebook walks through two critical debugging scenarios using trace data:
identifying failed tool invocations and detecting context window pressure.

**What you'll learn:**
- Find error spans and inspect exception details
- Understand how errors propagate through the trace hierarchy
- Detect context window pressure by tracking token growth
- Apply mitigation strategies for both scenarios

**Prerequisites:**
- Complete [01_tracing_setup.ipynb](01_tracing_setup.ipynb) and [02_trace_hierarchy.ipynb](02_trace_hierarchy.ipynb)

In [ ]:
import sys
sys.path.insert(0, ".")

from strands import Agent, tool
from strands.models.bedrock import BedrockModel
from strands.telemetry.config import StrandsTelemetry
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter, SpanExportResult
from opentelemetry.trace import StatusCode
from opentelemetry import trace

from trace_utils import format_trace_tree, find_error_spans, analyze_token_growth


class SpanCollector(SpanExporter):
    """Simple in-memory span collector for tutorial use."""
    def __init__(self):
        self._spans = []
    def export(self, spans):
        self._spans.extend(spans)
        return SpanExportResult.SUCCESS
    def get_finished_spans(self):
        return list(self._spans)
    def clear(self):
        self._spans = []
    def shutdown(self):
        self._spans = []


# Configure telemetry
telemetry = StrandsTelemetry()
telemetry.setup_console_exporter()

span_collector = SpanCollector()
provider = trace.get_tracer_provider()
if hasattr(provider, "add_span_processor"):
    provider.add_span_processor(SimpleSpanProcessor(span_collector))

print("✓ Telemetry configured")

## Scenario 1: Tool Failure Detection

When a tool raises an exception, the Strands SDK records:
- **Span status** → `ERROR`
- **Status description** → the error message
- **Exception event** → `exception.type`, `exception.message`, `exception.stacktrace`

The agent typically handles the error gracefully (reports it to the user or retries),
so the root Agent span may still be `OK` even when a tool span is `ERROR`.

In [ ]:
# Track call count to make failure deterministic
_flaky_api_call_count = 0


@tool
def flaky_api(query: str) -> str:
    """Simulate an unreliable external API call.

    Fails on the first call with a ConnectionError to demonstrate
    how tool errors appear in traces. Succeeds on subsequent calls
    to show agent recovery behavior.

    Args:
        query: The search query to send to the simulated API.

    Returns:
        A simulated API response string.

    Raises:
        ConnectionError: Simulated timeout on first call.
    """
    global _flaky_api_call_count
    _flaky_api_call_count += 1
    if _flaky_api_call_count == 1:
        raise ConnectionError(f"API timeout after 30s for query: {query}")
    return f"API result for: {query}"


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# Reset call counter so flaky_api fails on first call
_flaky_api_call_count = 0
span_collector.clear()

debug_agent = Agent(
    model=BedrockModel(model_id="us.amazon.nova-lite-v1:0"),
    tools=[calculator, flaky_api],
)

print("Invoking agent with flaky_api (expecting a tool failure)...\n")
result = debug_agent("Search for 'distributed tracing tutorial' using the API")
print(f"\nAgent response: {result}")

In [ ]:
# Use trace_utils to find error spans
spans = span_collector.get_finished_spans()
errors = find_error_spans(spans)

print(f"🔍 Found {len(errors)} error span(s):\n")

for span in errors:
    print(f"  ✗ Span: {span.name}")
    print(f"    Status code: {span.status.status_code.name}")
    attrs = span.attributes or {}
    if attrs.get("tool.status") == "error":
        print(f"    tool.status: error")
    if "gen_ai.tool.name" in attrs:
        print(f"    Tool: {attrs['gen_ai.tool.name']}")

    # Check for exception events
    for event in span.events:
        if event.name == "exception":
            print(f"    Exception type: {event.attributes.get('exception.type', 'N/A')}")
            print(f"    Exception msg:  {event.attributes.get('exception.message', 'N/A')}")
        # Also check tool result for error message
        if event.name == "gen_ai.choice":
            msg = str(event.attributes.get('message', ''))
            if 'Error:' in msg:
                print(f"    Error detail: {msg[:200]}")
    print()

if not errors:
    print("  No errors detected in this invocation.")

print("\n📋 Full trace tree:")
print(format_trace_tree(spans))

### Debugging Pattern: Error → Recovery

The key insight: filter by `status_code == ERROR` to find the failure, then look at
the *next* cycle's model invoke span to see how the agent recovered.

```
Agent Span (status: OK — agent handled it)
├── Cycle 1
│   ├── Model Invoke (OK) — decided to call flaky_api
│   └── Tool: flaky_api (ERROR) ← find this
│       exception.type: ConnectionError
├── Cycle 2 ← then look here
│   └── Model Invoke (OK) — "I encountered an error..."
```

**Production alert:** Set up monitoring on error span count. If a tool's error rate
exceeds 5% over 5 minutes, trigger an alert.

## Scenario 2: Context Window Pressure

Each cycle adds to the conversation history. When tools return large outputs,
token usage grows rapidly. Monitor `gen_ai.usage.input_tokens` across model
invoke spans to detect this.

**Warning signs:**
- Input tokens growing significantly between cycles
- Total approaching the model's context limit (200K for Claude)
- Degraded response quality

In [ ]:
@tool
def token_heavy(topic: str) -> str:
    """Generate a verbose response that consumes many tokens.

    Intentionally produces ~2000 tokens of output to demonstrate
    context window pressure across multiple agent cycles.

    Args:
        topic: The topic to generate content about.

    Returns:
        A lengthy string of repeated content.
    """
    paragraph = (
        f"Detailed analysis of {topic}: This is an extensive exploration covering "
        f"multiple dimensions and perspectives. The topic of {topic} encompasses "
        f"various interconnected aspects requiring thorough examination. "
        f"When considering {topic}, one must account for historical context, "
        f"current state, and future implications. "
    )
    return paragraph * 10

In [ ]:
span_collector.clear()

token_agent = Agent(
    model=BedrockModel(model_id="us.amazon.nova-lite-v1:0"),
    tools=[token_heavy, calculator],
)

print("Invoking agent with token_heavy tool (~2000 tokens per call)...\n")
result = token_agent(
    "Give me a detailed analysis of 'machine learning' and then 'distributed systems'. "
    "Use the token_heavy tool for each topic."
)
print(f"\nAgent response (truncated): {str(result)[:150]}...")

In [ ]:
# Use trace_utils to analyze token growth
spans = span_collector.get_finished_spans()
analysis = analyze_token_growth(spans)

print("📊 Token Usage Across Model Calls\n")
print(f"{'Span':<30} {'Input':<12} {'Output':<12}")
print("─" * 54)
for ms in analysis["model_spans"]:
    print(f"{ms['name']:<30} {ms['input_tokens']:<12} {ms['output_tokens']:<12}")

if "first_input" in analysis:
    print(f"\n⚠️  Context Growth Analysis:")
    print(f"   First model call: {analysis['first_input']} input tokens")
    print(f"   Last model call:  {analysis['last_input']} input tokens")
    print(f"   Growth: +{analysis['growth']} tokens ({analysis['growth_pct']:.0f}% increase)")
    print(f"   Context usage: {analysis['usage_pct']:.1f}% of {analysis['context_limit']:,} limit")

    if analysis["usage_pct"] > 50:
        print("   🚨 HIGH: Context window more than half full!")
    elif analysis["usage_pct"] > 20:
        print("   ⚠️  MODERATE: Monitor closely.")
    else:
        print("   ✓ LOW: Within safe limits.")

### Mitigation Strategies

| Strategy | When to Use |
|----------|-------------|
| Summarize tool outputs | Tools return verbose text |
| Limit cycle count | Agent might loop indefinitely |
| Use concise tools | Return structured data, not prose |
| Set alerts at 80% | Production monitoring |
| Use `SlidingWindowConversationManager` | Long-running conversations |

See [17-conversation-management](../17-conversation-management) for conversation
management strategies that help control context growth.

## Next

Continue to [04_backend_export.ipynb](04_backend_export.ipynb) to learn how to
export traces to production backends via OTLP.